# Policy iteration and value iteration: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

### 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)

In [ ]:
# 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)
rows = ["...G", ".#.X", "S..."]
R, C = len(rows), len(rows[0])
STEP, SLIP = -0.04, 0.2
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # up, right, down, left
ARROW = "^>v<"
cell = lambda s: rows[s // C][s % C]
terminal = lambda s: cell(s) in "GX"
wall = lambda s: cell(s) == "#"
reward = lambda s: {"G": 1.0, "X": -1.0}.get(cell(s), STEP)

def move(s, a):
    r, c = s // C + ACTIONS[a][0], s % C + ACTIONS[a][1]
    return s if not (0 <= r < R and 0 <= c < C) or rows[r][c] == "#" else r * C + c

def transitions(s, a):
    """list of (probability, next_state, reward)"""
    if terminal(s) or wall(s):
        return [(1.0, s, 0.0)]
    out = [(1 - SLIP, a), (SLIP / 2, (a + 1) % 4), (SLIP / 2, (a + 3) % 4)]
    return [(p, move(s, b), reward(move(s, b))) for p, b in out]

def q_value(s, a, V, gamma):
    return sum(p * (r + gamma * V[s2]) for p, s2, r in transitions(s, a))

def show(V):
    for i in range(R):
        print("  ".join(f"{V[i * C + j]:6.3f}" if not wall(i * C + j) else "  wall" for j in range(C)))

nS = R * C
live = [s for s in range(nS) if not terminal(s) and not wall(s)]

### 2. Value iteration

In [ ]:
# 2. Value iteration
def value_iteration(gamma, theta=1e-8):
    V = np.zeros(nS)
    for k in range(1, 10000):
        newV = V.copy()
        for s in live:
            newV[s] = max(q_value(s, a, V, gamma) for a in range(4))
        delta = np.abs(newV - V).max(); V = newV
        if delta < theta:
            return V, k

V, sweeps = value_iteration(0.9)
policy = {s: int(np.argmax([q_value(s, a, V, 0.9) for a in range(4)])) for s in live}
print("value iteration converged after", sweeps, "sweeps")
show(V)
print("greedy policy:")
for i in range(R):
    print(" ".join(ARROW[policy[i * C + j]] if (i * C + j) in policy else ("G" if cell(i * C + j) == "G" else "X" if cell(i * C + j) == "X" else "#") for j in range(C)))

### 3. Policy iteration

In [ ]:
# 3. Policy iteration
def policy_iteration(gamma):
    pi = {s: 0 for s in live}                         # start with 'always up'
    for it in range(1, 50):
        V = np.zeros(nS)
        for _ in range(10000):                        # policy evaluation
            newV = V.copy()
            for s in live:
                newV[s] = q_value(s, pi[s], V, gamma)
            done = np.abs(newV - V).max() < 1e-10; V = newV
            if done:
                break
        new_pi = {s: int(np.argmax([q_value(s, a, V, gamma) for a in range(4)])) for s in live}   # improvement
        if new_pi == pi:
            return V, pi, it
        pi = new_pi

V2, pi2, iters = policy_iteration(0.9)
print("policy iteration stable after", iters, "improvement rounds; same policy as value iteration:", pi2 == policy)
print("max |V_vi - V_pi| =", float(np.abs(V - V2).max()))

### 4. Effect of the discount factor on the number of sweeps

In [ ]:
# 4. Effect of the discount factor on the number of sweeps
for g in (0.5, 0.9, 0.99):
    print(f"gamma = {g}: value iteration needs {value_iteration(g)[1]} sweeps")

## Expected output

Running all cells above should print:

```
value iteration converged after 26 sweeps
 0.610   0.766   0.928   0.000
 0.487    wall   0.585   0.000
 0.374   0.327   0.428   0.189
greedy policy:
> > > G
^ # ^ X
^ > ^ <
policy iteration stable after 3 improvement rounds; same policy as value iteration: True
max |V_vi - V_pi| = 5.343922915290733e-09
gamma = 0.5: value iteration needs 20 sweeps
gamma = 0.9: value iteration needs 26 sweeps
gamma = 0.99: value iteration needs 32 sweeps
```

## Practice exercises

Try these in the cells below.

1. Count how many sweeps value iteration needs for gamma = 0.5, 0.9, 0.99 with theta = 1e-6.
2. Make the environment slippery (slip = 0.4). Does the optimal policy change on the classic grid?
3. Implement modified policy iteration: only 3 evaluation sweeps per improvement.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
